<a href="https://colab.research.google.com/github/jerome10095/100-javascript-projects/blob/master/kinyarwanda_voice_assistant_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kinyarwanda Voice Assistant — Model Test Lab

This notebook tests the three models we picked, then joins them into one voice assistant you can talk to.

| Step | Model | License |
|---|---|---|
| Speech-to-text | `badrex/w2v-bert-2.0-kinyarwanda-asr` | CC-BY-4.0 |
| Chat | `google/gemma-4-E4B-it` (switchable to `gemma-4-12B-it`) | Apache-2.0 |
| Text-to-speech | `Sunbird/orpheus-3b-tts-multilingual` | Apache-2.0 (built on Llama 3.2) |

**Before you start:** in Colab go to **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom (Shift+Enter).

The first run downloads about 15 GB of models, so it takes 10–15 minutes. Later runs in the same session are fast.

## 1. Install libraries

In [1]:
!pip install -q -U transformers accelerate bitsandbytes snac librosa soundfile gradio
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU!")

GPU: Tesla T4


## 2. Settings
Change the models here if you want to compare alternatives. On the free T4 GPU keep `gemma-4-E4B-it`; `gemma-4-12B-it` gives better answers but needs a bigger GPU (Colab Pro L4/A100).

In [2]:
ASR_MODEL  = "badrex/w2v-bert-2.0-kinyarwanda-asr"   # alt: "mbazaNLP/Whisper-Small-Kinyarwanda"
CHAT_MODEL = "google/gemma-4-E4B-it"                 # upgrade: "google/gemma-4-12B-it"
TTS_MODEL  = "Sunbird/orpheus-3b-tts-multilingual"

# Orpheus-style models pick a voice with a prefix like "<speaker>: text".
# Check the Sunbird model card for the Kinyarwanda speaker name/ID and put it here.
# Leave "" to let the model choose.
TTS_SPEAKER = "bateesa_kin_0001"   # Kinyarwanda speaker from the Sunbird model card

SYSTEM_PROMPT = (
    "Uri umufasha w'ubwenge bw'ubukorano uvuga Ikinyarwanda. "
    "Subiza buri gihe mu Kinyarwanda cyiza kandi gisobanutse, mu nteruro ngufi. "
    "You are a helpful assistant. Always answer in clear, natural Kinyarwanda, in short sentences."
)
DEVICE = "cuda"

## 3. Speech-to-text (ASR)

In [3]:
import librosa, numpy as np
from transformers import pipeline

asr = pipeline("automatic-speech-recognition", model=ASR_MODEL,
               device=0, torch_dtype=torch.float16)

def transcribe(audio_path):
    wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    out = asr({"raw": wav, "sampling_rate": 16000}, chunk_length_s=20, stride_length_s=4)
    return out["text"].strip()

print("ASR ready")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/775 [00:00<?, ?it/s]

ASR ready


### Test it
Upload a short Kinyarwanda recording (WhatsApp voice notes work: `.ogg`, `.mp3`, `.wav`, `.m4a`) using the folder icon on the left, then put its name below.

In [4]:
TEST_AUDIO = "test.ogg"   # <- change to your file name
import os
if os.path.exists(TEST_AUDIO):
    print("Transcript:", transcribe(TEST_AUDIO))
else:
    print("Upload a file first, then rerun this cell.")

Upload a file first, then rerun this cell.


## 4. Chat model (Gemma 4)

In [5]:
import transformers
from transformers import AutoProcessor, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(CHAT_MODEL)

chat_model, last_err = None, None
for cls_name in ["AutoModelForImageTextToText", "AutoModelForMultimodalLM", "AutoModelForCausalLM"]:
    cls = getattr(transformers, cls_name, None)
    if cls is None:
        continue
    try:
        chat_model = cls.from_pretrained(CHAT_MODEL, quantization_config=bnb,
                                         device_map=DEVICE, torch_dtype=torch.float16)
        print("Loaded chat model with", cls_name)
        break
    except Exception as e:
        last_err = e
if chat_model is None:
    raise last_err

history = []   # keeps the conversation

def _build(messages, structured=True):
    if structured:
        return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]
    return messages

def chat(user_text, max_new_tokens=300, remember=True):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history + [{"role": "user", "content": user_text}]
    try:
        inputs = processor.apply_chat_template(_build(messages), add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt")
    except Exception:
        inputs = processor.apply_chat_template(_build(messages, False), add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt")
    inputs = inputs.to(chat_model.device)
    with torch.inference_mode():
        out = chat_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                  temperature=0.7, top_p=0.95)
    reply = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    if remember:
        history.extend([{"role": "user", "content": user_text}, {"role": "assistant", "content": reply}])
    return reply

def reset_chat():
    history.clear()

print(chat("Muraho! Wambwira mu nteruro ebyiri icyo ushoboye gukora?", remember=False))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Loaded chat model with AutoModelForImageTextToText
Muraho cyane! Ndashobora kugufasha mu byindi bikoreshwa. Nshobora kugufasha mu kumenya ibintu byose.


### Kinyarwanda quality check
Read each answer and score it 1–5 for **correct Kinyarwanda**, **natural wording** and **useful content**. This is the riskiest part of the project, so be strict.

In [6]:
test_questions = [
    "Ni gute nakwita ku kigori kugira ngo nsarure byinshi?",
    "Nsobanurira muri make uko umusoro ku nyongeragaciro (TVA) ukora mu Rwanda.",
    "Ndashaka kwandika ibaruwa isaba akazi. Wamfasha gute?",
    "Ubwenge bw'ubukorano ni iki? Bisobanure nk'aho uvugana n'umwana w'imyaka 12.",
    "Ni izihe nama watanga ku muntu utangiye ubucuruzi buto i Kigali?",
]
for q in test_questions:
    print("Q:", q)
    print("A:", chat(q, remember=False))
    print("-" * 80)

Q: Ni gute nakwita ku kigori kugira ngo nsarure byinshi?
A: Kugira ngo usarure byinshi, ugomba kwita ku ibintu byiza.

Niba ugiye ku kigori, menya ibyo wumva.
--------------------------------------------------------------------------------
Q: Nsobanurira muri make uko umusoro ku nyongeragaciro (TVA) ukora mu Rwanda.
A: Uyu mwanzu ni ubuhanga butanga amakuru. Nta kintu cyo kubwira cyangwa gutanga.
--------------------------------------------------------------------------------
Q: Ndashaka kwandika ibaruwa isaba akazi. Wamfasha gute?
A: Nta kibazo. Twakora ibyo. Ni iki kintu cy'akazi uzi?
--------------------------------------------------------------------------------
Q: Ubwenge bw'ubukorano ni iki? Bisobanure nk'aho uvugana n'umwana w'imyaka 12.
A: Ubwenge bw'ubukorano ni uburyo tubundi mu buzima. Ni kumenya ibintu byose mu buzima. Ni kubona ibintu mu buryo bw'ubwo bw'ubwitonzi.
--------------------------------------------------------------------------------
Q: Ni izihe nama watanga ku 

In [7]:
# Stage 1 - Option A: translate around Gemma (Kinyarwanda -> English -> Gemma -> Kinyarwanda)
# Translators: Helsinki-NLP OPUS-MT rw->en and en->rw (Apache-2.0, commercial use OK).
# Small (~300 MB each), so they run on the CPU and use no GPU memory (the T4 is already full).
# Note: trained mostly on JW300 (religious texts) + Tatoeba, so wording may sound formal - fine-tuning later fixes that.
from transformers import MarianMTModel, MarianTokenizer
MT = {}
for tgt, name in [("en", "Helsinki-NLP/opus-mt-rw-en"), ("rw", "Helsinki-NLP/opus-mt-en-rw")]:
    MT[tgt] = (MarianTokenizer.from_pretrained(name), MarianMTModel.from_pretrained(name).eval())

def translate(text, target="en", max_new_tokens=256):
    tok, mdl = MT[target]
    # translate sentence by sentence: these small models handle short inputs best
    import re
    parts = [p for p in re.split(r'(?<=[.!?])\s+', " ".join(text.split())) if p]
    batch = tok(parts, return_tensors="pt", padding=True, truncation=True)
    with torch.inference_mode():
        out = mdl.generate(**batch, max_new_tokens=max_new_tokens, num_beams=4)
    return " ".join(tok.batch_decode(out, skip_special_tokens=True)).strip()

print(translate("Muraho! Amakuru yawe?", "en"), "|", translate("Hello, how are you today?", "rw"))

SYSTEM_PROMPT_EN = ("You are a helpful, knowledgeable assistant for people in Rwanda. "
                    "Answer clearly and correctly in 2-4 short, simple sentences. "
                    "Use plain words, because your answer will be translated into Kinyarwanda.")
en_history = []

def _gemma(messages, max_new_tokens=300):
    try:
        inputs = processor.apply_chat_template(_build(messages), add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt")
    except Exception:
        inputs = processor.apply_chat_template(_build(messages, False), add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt")
    inputs = inputs.to(chat_model.device)
    with torch.inference_mode():
        out = chat_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def chat_translate(user_text, max_new_tokens=300, remember=True, show_steps=False):
    q_en = translate(user_text, "en")
    messages = [{"role": "system", "content": SYSTEM_PROMPT_EN}] + en_history + [{"role": "user", "content": q_en}]
    a_en = _gemma(messages, max_new_tokens)
    a_rw = translate(" ".join(a_en.split()), "rw", max_new_tokens=400)
    if remember:
        en_history.extend([{"role": "user", "content": q_en}, {"role": "assistant", "content": a_en}])
    if show_steps:
        print("   [EN question]", q_en); print("   [EN answer]  ", a_en)
    return a_rw

# Side-by-side test on the same 5 questions
for q in test_questions:
    print("Q:", q)
    print("A (translate):", chat_translate(q, remember=False, show_steps=True))
    print("-" * 80)

# Pick which chat the app uses: "translate" or "direct". The running app picks this up right away.
if "chat_direct" not in globals():
    chat_direct = chat
CHAT_MODE = "translate"
chat = chat_translate if CHAT_MODE == "translate" else chat_direct
def reset_chat():
    history.clear(); en_history.clear()
print("App chat mode:", CHAT_MODE)

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/810k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.33M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


...

   [EN question] I would like to write a letter asking for work. How can you help me?
   [EN answer]   I can help you write a strong letter. Tell me what kind of job you want. We can make it clear and professional.
A (translate): Nshobora kugufasha kwandika ibaruwa ikomeye. Mbwira akazi wifuza gukora. Dushobora kuwusobanukirwa neza kandi tukamutoza umwuga.
--------------------------------------------------------------------------------
Q: Ubwenge bw'ubukorano ni iki? Bisobanure nk'aho uvugana n'umwana w'imyaka 12.


[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [EN question] What is wisdom? It is as if you are talking to a 12 - year - old.
   [EN answer]   Wisdom is knowing what is right and wrong. It means using your knowledge to make good choices. Wise people think carefully about things before they act.
A (translate): Ubwenge ni ukumenya icyiza n'ikibi. Bisobanura gukoresha ubumenyi bwawe kugira ngo ufate imyanzuro myiza. Abantu b'abanyabwenge babanza gutekereza babigiranye ubwitonzi ku bintu mbere yo kugira icyo bakora.
--------------------------------------------------------------------------------
Q: Ni izihe nama watanga ku muntu utangiye ubucuruzi buto i Kigali?


[transformers] Both `max_new_tokens` (=400) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   [EN question] What suggestions could you give about someone who began small business in the Scriptures?
   [EN answer]   The Bible shows that God blesses hard work. Keep working diligently and faithfully in your business. Trust in God's provision for your growth.
A (translate): Bibiliya igaragaza ko Imana iha imigisha umurimo ukoranywe umwete. Jya ukorana umwete kandi ube uwizerwa mu kazi ukora. Jya wiringira uburyo Imana yateganyije bwo gukura.
--------------------------------------------------------------------------------
App chat mode: translate


## 5. Text-to-speech (Orpheus + SNAC decoder)

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from snac import SNAC
from IPython.display import Audio, display

tts_tok = AutoTokenizer.from_pretrained(TTS_MODEL)
import gc; gc.collect(); torch.cuda.empty_cache()
# 4-bit so it fits on the T4 next to Gemma (fp16 needs ~6.6 GB and runs out of memory)
tts_model = AutoModelForCausalLM.from_pretrained(TTS_MODEL, quantization_config=bnb, device_map=DEVICE, torch_dtype=torch.float16).eval()
snac = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to(DEVICE).eval()

# Standard Orpheus special tokens
START_HUMAN, END_TEXT, END_HUMAN = 128259, 128009, 128260
START_SPEECH, END_SPEECH, AUDIO_OFFSET = 128257, 128258, 128266

def _codes_to_audio(codes):
    codes = codes[: len(codes) // 7 * 7]
    l1, l2, l3 = [], [], []
    for i in range(len(codes) // 7):
        c = codes[7 * i: 7 * i + 7]
        f = [c[0], c[1] - 4096, c[2] - 2 * 4096, c[3] - 3 * 4096,
             c[4] - 4 * 4096, c[5] - 5 * 4096, c[6] - 6 * 4096]
        if any(x < 0 or x >= 4096 for x in f):
            continue  # skip a broken frame instead of crashing
        l1.append(f[0]); l2 += [f[1], f[4]]; l3 += [f[2], f[3], f[5], f[6]]
    if not l1:
        return np.zeros(2400, dtype=np.float32)
    layers = [torch.tensor(l, device=DEVICE).unsqueeze(0) for l in (l1, l2, l3)]
    with torch.inference_mode():
        audio = snac.decode(layers)
    return audio.squeeze().float().cpu().numpy()

def speak(text, speaker=None, max_new_tokens=2000):
    speaker = TTS_SPEAKER if speaker is None else speaker
    prompt = f"{speaker}: {text}" if speaker else text
    ids = tts_tok(prompt, return_tensors="pt").input_ids
    ids = torch.cat([torch.tensor([[START_HUMAN]]), ids, torch.tensor([[END_TEXT, END_HUMAN]])], dim=1).to(DEVICE)
    with torch.inference_mode():
        out = tts_model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=max_new_tokens,
                                 do_sample=True, temperature=0.6, top_p=0.95,
                                 repetition_penalty=1.1, eos_token_id=END_SPEECH)
    gen = out[0, ids.shape[1]:].tolist()
    if START_SPEECH in gen:
        gen = gen[len(gen) - gen[::-1].index(START_SPEECH):]
    codes = [t - AUDIO_OFFSET for t in gen if t >= AUDIO_OFFSET]
    return _codes_to_audio(codes), 24000

wav, sr = speak("Muraho neza! Nitwa umufasha wawe w'ubwenge bw'ubukorano. Nishimiye kugufasha uyu munsi.")
display(Audio(wav, rate=sr))

config.json:   0%|          | 0.00/915 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.41M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 22.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/508 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 79.5MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

### Voice quality check
Listen for: correct pronunciation, a natural Rwandan accent, and no cut-off or mumbling. If it sounds wrong, the most likely fix is setting `TTS_SPEAKER` in step 2 to the Kinyarwanda speaker listed on the model card.

In [9]:
for s in ["Amakuru yawe? Uyu munsi ikirere kimeze neza i Kigali.",
          "Murakoze cyane kuba mwaduhisemo. Tubifurije umunsi mwiza.",
          "Igiciro cy'ibishyimbo ku isoko rya Kimironko cyazamutseho amafaranga magana abiri."]:
    w, r = speak(s); print(s); display(Audio(w, rate=r))

Amakuru yawe? Uyu munsi ikirere kimeze neza i Kigali.


Murakoze cyane kuba mwaduhisemo. Tubifurije umunsi mwiza.


Igiciro cy'ibishyimbo ku isoko rya Kimironko cyazamutseho amafaranga magana abiri.


In [10]:
# Option B: Meta MMS Kinyarwanda voice (facebook/mms-tts-kin)
# Small (~150 MB) VITS model trained on Kinyarwanda only. Compare it with Orpheus above.
from transformers import VitsModel, AutoTokenizer as _AT
mms_tok = _AT.from_pretrained("facebook/mms-tts-kin")
mms_model = VitsModel.from_pretrained("facebook/mms-tts-kin").to(DEVICE).eval()

def speak_mms(text, speaker=None, **kw):
    ids = mms_tok(text.lower(), return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        wav = mms_model(**ids).waveform[0].float().cpu().numpy()
    return wav, mms_model.config.sampling_rate

for s in ["Amakuru yawe? Uyu munsi ikirere kimeze neza i Kigali.",
          "Murakoze cyane kuba mwaduhisemo. Tubifurije umunsi mwiza.",
          "Igiciro cy'ibishyimbo ku isoko rya Kimironko cyazamutseho amafaranga magana abiri."]:
    w, r = speak_mms(s); print("MMS:", s); display(Audio(w, rate=r))

# Pick the voice the app uses: "orpheus" or "mms". The running app picks this up right away.
if "speak_orpheus" not in globals():
    speak_orpheus = speak
TTS_ENGINE = "orpheus"
speak = speak_mms if TTS_ENGINE == "mms" else speak_orpheus
print("App voice:", TTS_ENGINE)

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  145MB            

...

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

MMS: Amakuru yawe? Uyu munsi ikirere kimeze neza i Kigali.


MMS: Murakoze cyane kuba mwaduhisemo. Tubifurije umunsi mwiza.


MMS: Igiciro cy'ibishyimbo ku isoko rya Kimironko cyazamutseho amafaranga magana abiri.


App voice: orpheus


## 6. Full voice assistant: speak → transcribe → answer → speak back

In [11]:
import time

def voice_turn(audio_path):
    t0 = time.time(); heard = transcribe(audio_path)
    t1 = time.time(); reply = chat(heard)
    t2 = time.time(); wav, sr = speak(reply)
    t3 = time.time()
    timing = f"ASR {t1-t0:.1f}s | Chat {t2-t1:.1f}s | TTS {t3-t2:.1f}s | Total {t3-t0:.1f}s"
    return heard, reply, (sr, wav), timing

if os.path.exists(TEST_AUDIO):
    heard, reply, (sr, wav), timing = voice_turn(TEST_AUDIO)
    print("You said:", heard); print("Assistant:", reply); print(timing)
    display(Audio(wav, rate=sr))

## 7. Test app (microphone + text)
Run this cell and open the **public link** it prints (`...gradio.live`). It works on your phone too, so you can let friends or family test it. The link expires after 72 hours or when Colab stops.

In [12]:
import gradio as gr

def from_mic(audio_path):
    if not audio_path:
        return "", "No audio received.", None, ""
    heard, reply, audio, timing = voice_turn(audio_path)
    return heard, reply, audio, timing

def from_text(text):
    t0 = time.time(); reply = chat(text); t1 = time.time()
    wav, sr = speak(reply); t2 = time.time()
    return reply, (sr, wav), f"Chat {t1-t0:.1f}s | TTS {t2-t1:.1f}s"

with gr.Blocks(title="Kinyarwanda Voice Assistant") as app:
    gr.Markdown("# Umufasha w'Ikinyarwanda\nVuga cyangwa wandike mu Kinyarwanda.")
    with gr.Tab("Voice"):
        mic = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Vuga hano")
        btn = gr.Button("Ohereza", variant="primary")
        heard = gr.Textbox(label="Ibyo wavuze (transcript)")
        reply = gr.Textbox(label="Igisubizo")
        voice = gr.Audio(label="Igisubizo mu majwi", autoplay=True)
        timing = gr.Textbox(label="Speed")
        btn.click(from_mic, mic, [heard, reply, voice, timing])
    with gr.Tab("Text"):
        txt = gr.Textbox(label="Andika ikibazo")
        btn2 = gr.Button("Ohereza", variant="primary")
        reply2 = gr.Textbox(label="Igisubizo")
        voice2 = gr.Audio(label="Igisubizo mu majwi", autoplay=True)
        timing2 = gr.Textbox(label="Speed")
        btn2.click(from_text, txt, [reply2, voice2, timing2])
    gr.Button("New conversation").click(lambda: reset_chat())

app.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://722d400b123ddb1cb6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 8. What to record from your tests
Keep simple notes (a Google Sheet is fine) for each test:

- **ASR:** what you said vs. what it wrote (count wrong words). Test quiet rooms, noise, fast speech, different speakers, and mixed Kinyarwanda/English.
- **Chat:** your 1–5 scores from step 4.
- **TTS:** 1–5 for accent and clarity.
- **Speed:** the timing line. Under ~5 seconds total feels conversational.

These notes decide what to fine-tune first. Most likely that's the chat model's Kinyarwanda, then the voice.

**If you run out of GPU memory:** Runtime → Restart session, then switch `CHAT_MODEL` back to `gemma-4-E4B-it`.